# 1주차 SQL/EDA 실습 노트북

이 노트북은 `stt_sign_pple_mm.sqlite`를 조회하면서 핵심 지표를 점검하는 실습용 자료입니다.

## 0) 준비
- 노트북 위치: `1주차/02_SQL실습/notebooks`
- DB 위치: `data/db/stt_sign_pple_mm.sqlite`
- 커널: Python 3

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

db_path = (Path.cwd() / '../../../data/db/stt_sign_pple_mm.sqlite').resolve()
print('DB path :', db_path)
print('Exists  :', db_path.exists())
conn = sqlite3.connect(db_path)


In [ ]:
def run_query(sql, n=10):
    cur = conn.cursor()
    cur.execute(sql)
    cols = [d[0] for d in cur.description] if cur.description else []
    rows = cur.fetchmany(n) if cur.description else []
    print('Columns:', cols)
    print('Rows   :', len(rows))
    for r in rows:
        print(r)

def run_query_df(sql, limit=20):
    # pandas DataFrame ??? ??? ?? ??
    q = f"SELECT * FROM ({sql.rstrip().rstrip(';')}) LIMIT {int(limit)}"
    df = pd.read_sql_query(q, conn)
    display(df)
    return df

def amount_expr(col_name):
    return f"CAST(NULLIF(REPLACE(REPLACE(REPLACE(COALESCE(\"{col_name}\", ''), char(8361), ''), ',', ''), ' ', ''), '') AS REAL)"


## 1) 테이블/컬럼 확인

In [ ]:
run_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", n=20)


In [ ]:
run_query('PRAGMA table_info("stt_sign_pple_mm");', n=20)


## 2) 샘플 데이터 보기

In [ ]:
sql = '''
SELECT
  "????(Real>String)" AS accident_no,
  "????(Integer>String)" AS policy_no,
  "?????" AS coverage_name,
  "????" AS disease_code,
  "??????(Integer>Currency)" AS claim_amt_raw,
  "????(Integer>Currency)" AS decided_amt_raw
FROM "stt_sign_pple_mm"
LIMIT 20;
'''

# 1) ??? ?? ??
run_query(sql, n=10)

# 2) pandas ?(DataFrame) ??
df_sample = run_query_df(sql, limit=20)
df_sample


## 3) 결정금액 빈도 Top 10

In [ ]:
dec_expr = amount_expr('결정금액(Integer>Currency)')
sql = f'''
SELECT {dec_expr} AS decided_amt, COUNT(*) AS cnt
FROM "stt_sign_pple_mm"
WHERE {dec_expr} IS NOT NULL
GROUP BY decided_amt
ORDER BY cnt DESC, decided_amt ASC
LIMIT 10;
'''
run_query(sql, n=10)


## 4) 이상치 건수 (IQR 상한 690000)

In [ ]:
sql = f'''
SELECT COUNT(*) AS outlier_cnt
FROM "stt_sign_pple_mm"
WHERE {dec_expr} > 690000;
'''
run_query(sql, n=10)


## 5) 질병코드 빈도 Top 10

In [ ]:
sql = '''
SELECT
  "질병코드" AS disease_code,
  "질병명" AS disease_name,
  COUNT(*) AS claim_lines
FROM "stt_sign_pple_mm"
WHERE COALESCE("질병코드", '') <> ''
GROUP BY "질병코드", "질병명"
ORDER BY claim_lines DESC
LIMIT 10;
'''
run_query(sql, n=10)


## 6) 청구일 -> 지급일 지연일수 분포

In [ ]:
sql = '''
SELECT
  CAST(julianday("지급년월일") - julianday("사고청구년월일") AS INTEGER) AS lag_days,
  COUNT(*) AS cnt
FROM "stt_sign_pple_mm"
WHERE length("사고청구년월일") = 10
  AND length("지급년월일") = 10
  AND julianday("지급년월일") >= julianday("사고청구년월일")
GROUP BY lag_days
ORDER BY cnt DESC
LIMIT 10;
'''
run_query(sql, n=10)


## 7) 마무리

In [ ]:
conn.close()
print('Connection closed')


## 6-1) ???
`pandas` + `matplotlib`? ?? ?? 2?? ?????.

In [ ]:
# ???? ?????
hist_sql = f'''
SELECT {dec_expr} AS decided_amt
FROM "stt_sign_pple_mm"
WHERE {dec_expr} IS NOT NULL AND {dec_expr} > 0
LIMIT 200000
'''
df_hist = run_query_df(hist_sql, limit=200000)
plt.figure(figsize=(8,4))
plt.hist(df_hist['decided_amt'], bins=50)
plt.title('Decided Amount Histogram')
plt.xlabel('decided_amt')
plt.ylabel('count')
plt.tight_layout()
plt.show()


In [ ]:
# ???? ?? 10? ????
bar_sql = '''
SELECT "????" AS disease_code, COUNT(*) AS claim_lines
FROM "stt_sign_pple_mm"
WHERE COALESCE("????", '') <> ''
GROUP BY "????"
ORDER BY claim_lines DESC
LIMIT 10
'''
df_bar = run_query_df(bar_sql, limit=10)
plt.figure(figsize=(9,4))
plt.bar(df_bar['disease_code'], df_bar['claim_lines'])
plt.title('Top 10 Disease Codes by Claim Lines')
plt.xlabel('disease_code')
plt.ylabel('claim_lines')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
